# Name That LLM — checkpoint 3 November 2026

Laporan yang dapat dijalankan ulang: EDA, audit validasi/cache, eksperimen CPU,
ablation, serta ekspektasi versus skor submission resmi. Target riset 0,99 belum
tercapai. Snapshot skor resmi terbaik akun adalah **0,665**. Notebook ini memakai
hasil eksperimen yang sudah disimpan; tidak membutuhkan GPU atau membuat submission.

Jalankan dari root snapshot source repo. Raw train tidak didistribusikan lewat GitHub;
letakkan data yang diunduh secara sah di `data/` atau attach competition di Kaggle.

In [1]:
import json, time
from contextlib import contextmanager
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p/'research/experiments/e15_reader_foldlocal/results.json').exists()), None)
assert ROOT is not None, 'Jalankan dari root source snapshot atau folder notebook di dalamnya.'
@contextmanager
def step(name, eta_seconds):
    begin = time.perf_counter()
    print(f'{name} | estimasi {eta_seconds}s', flush=True)
    yield
    print(f'{name} | selesai {time.perf_counter()-begin:.2f}s', flush=True)

## 1. EDA: unit observasi dan generalisasi
Split harus memisahkan prompt. Angka 70/15/15 diterapkan pada prompt dan
dibulatkan menjadi 105/22/23 dari 150 prompt. Lima blok test mencakup 115 prompt,
bukan 150; ini lima rotasi holdout, bukan test coverage 100% dari 5-fold biasa.

In [2]:
with step('EDA', 2):
    local = ROOT/'data/train.csv'
    kaggle = Path('/kaggle/input/competitions/name-that-llm/train.csv')
    path = local if local.exists() else kaggle
    if path.exists():
        train = pd.read_csv(path)
        assert {'id','text','model','prompt_id'}.issubset(train.columns)
        assert train.id.is_unique and train.notna().all().all()
        print('rows',len(train),'prompts',train.prompt_id.nunique(),'labels',train.model.nunique())
        display(train.model.value_counts().rename('support').to_frame())
        print('words:',train.text.str.split().str.len().describe().round(2).to_dict())
        markers = pd.DataFrame({k: train.text.str.contains(v, regex=False)
                                for k,v in {'em_dash':'—','curly_apostrophe':'’','semicolon':';'}.items()})
        markers['model'] = train.model
        display(markers.groupby('model').mean().round(3))
    else:
        print('Raw train belum dipasang. Ringkasan audit:2699 rows,150 prompts,18 labels;149/150 per kelas.')

EDA | estimasi 2s
rows 2699 prompts 150 labels 18
words: {'count': 2699.0, 'mean': 114.66, 'std': 58.28, 'min': 36.0, '25%': 64.0, '50%': 101.0, '75%': 157.0, 'max': 274.0}
EDA | selesai 0.07s


,support
model,
qwen3.8-max,150
claude-sonnet-5,150
gemini-3.6-flash,150
gpt-5.6-terra,150
glm-5.3-flash,150
gemini-3.1-pro-preview,150
gemini-3.8-flash,150
claude-opus-4-6,150
gpt-5.6-luna,150


,em_dash,curly_apostrophe,semicolon
model,,,
claude-fable-5,0.832,0.000,0.336
claude-opus-4-6,0.753,0.000,0.153
claude-opus-4-8,0.627,0.000,0.240
claude-opus-5,0.813,0.000,0.540
claude-sonnet-5,0.733,0.000,0.233
gemini-3.1-pro-preview,0.000,0.060,0.193
gemini-3.6-flash,0.000,0.193,0.227
gemini-3.7-flash,0.007,0.307,0.233
gemini-3.8-flash,0.000,0.247,0.280


## 2. Audit yang mengubah interpretasi skor

- OOF LoRA lama dipindahkan ke rotasi 70/15/15. Setiap baris tetap out-of-prompt,
  tetapi proses stacking bukan nested training yang sepenuhnya terpisah.
- Standardisasi/PCA lama dilatih global. E15/E16 melatih transformasi hanya di train.
- Pool yang direkonstruksi tidak cocok urutan labelnya dengan cache lama (91,33%
  posisi berbeda). Ini bukan bukti label cache salah. E15/E16 memakai fitur host
  saja; provenance pool harus dipulihkan sebelum dipakai lagi.
- Tidak ada hasil hidden per kelas yang tersedia. Diagnostik per kelas di sini
  berasal dari gold lokal, sedangkan leaderboard hanya metrik agregat.

In [3]:
with step('muat manifest', 1):
    manifests={}
    for name in ['e15_reader_foldlocal','e16_topic_projection']:
        m=json.loads((ROOT/'research/experiments'/name/'manifest.json').read_text())
        manifests[name]=m
        print(name,'signature',m['signature'],'split',m['split_md5'])
    assert manifests['e15_reader_foldlocal']['split_md5']==manifests['e16_topic_projection']['split_md5']
    print('External pool used in E15: False; alignment audit:',manifests['e15_reader_foldlocal']['reconstructed_pool_alignment'])

muat manifest | estimasi 1s
e15_reader_foldlocal signature 5ff79f4bfd9cc871e27f58c37490a9e8276b21824cb41d744822b561163465d2 split e52bf922570b1d00a0e51c510ac4f16d
e16_topic_projection signature cdb0a6989fb3914484b666d0885a487c864ad17d9dd104c616d4cb2e527d9d65 split e52bf922570b1d00a0e51c510ac4f16d
External pool used in E15: False; alignment audit: False
muat manifest | selesai 0.00s


## 3. E15: mengganti scorer pembaca frozen
Uji Qwen8/Qwen27 layer16/27, train-only scaler/PCA128, LR dua C dan shrinkage-LDA.
Pilih kandidat hanya pada validation. Anchor fixed:Qwen8-layer16 LR C0,003.
Ini perbandingan host-only; tidak setara dengan ensemble publik yang memakai pool besar.

In [4]:
with step('E15 hasil dan log', 1):
    e15=json.loads((ROOT/'research/experiments/e15_reader_foldlocal/results.json').read_text())
    folds=json.loads((ROOT/'research/experiments/e15_reader_foldlocal/folds.json').read_text())
    display(pd.DataFrame([dict(fold=r['fold'],model=r['chosen']['name'],train=r['chosen']['train_ba'],
                              val=r['chosen']['validation_ba'],test=r['selected_test_ba'],
                              anchor=r['anchor_test_ba'],seconds=r['elapsed_s']) for r in folds]).round(4))
    print('mean test',e15['selected_test_ba_mean'],'paired delta CI95',e15['paired_delta_ci95'])
    print('Keputusan:belum terkonfirmasi; tidak memenuhi submission gate.')

E15 hasil dan log | estimasi 1s
mean test 0.3546113306982872 paired delta CI95 [-0.011111111111111127, 0.0338675018956908]
Keputusan:belum terkonfirmasi; tidak memenuhi submission gate.
E15 hasil dan log | selesai 0.00s


,fold,model,train,val,test,anchor,seconds
0,0,q27_l16_lr_C0.003,0.6053,0.3917,0.3502,0.3333,2.1908
1,1,q27_l16_lr_C0.003,0.5984,0.3712,0.4059,0.3866,2.0778
2,2,q27_l16_lr_C0.003,0.6176,0.3737,0.3430,0.3237,2.0587
3,3,q27_l16_lr_C0.003,0.6187,0.3535,0.3937,0.3599,1.9542
4,4,q8_l27_lda_auto,0.4943,0.3510,0.2802,0.3140,2.0174


## 4. E16 dan E17: fusion versus proyeksi topik
E16 mempelajari arah nuisance dari centroid prompt training dalam PCA space,
kemudian memilih rank/gamma/C dan kekuatan fusion dengan SVC di validation.
E17 memisahkan manfaat reader fusion dari manfaat proyeksi topik itu sendiri.
Cached SVC lolos cek shape, finite validation/test dan NaN training mask;
provenance lengkap cache lama tetap lebih lemah daripada training baru dari manifest.

In [5]:
with step('E16/E17 ablation', 1):
    e16=json.loads((ROOT/'research/experiments/e16_topic_projection/results.json').read_text())
    e17=json.loads((ROOT/'research/experiments/e17_topic_ablation/results.json').read_text())
    display(pd.DataFrame({'test_mean':e17['test_mean'],'test_pooled':e17['test_pooled']}).round(4))
    display(pd.DataFrame([dict(comparison=k,ci95_low=v[0],ci95_high=v[1])
                          for k,v in e17['paired_ci95'].items()]).round(4))
    print('E16 positive rotations',e16['positive_rotations'],'/5')
    print('CPU seconds:E15',round(e15['elapsed_s'],2),'E16',round(e16['elapsed_s'],2),'E17',round(e17['elapsed_s'],2))
    print('Fusion membantu SVC; efek tambahan proyeksi belum terkonfirmasi jika CI melintasi0.')
    print('Skor cabang ini tidak mengalahkan champion resmi0.665; tidak membuat submission.')

E16/E17 ablation | estimasi 1s
E16 positive rotations 5 /5
CPU seconds:E15 13.28 E16 9.92 E17 4.8
Fusion membantu SVC; efek tambahan proyeksi belum terkonfirmasi jika CI melintasi0.
Skor cabang ini tidak mengalahkan champion resmi0.665; tidak membuat submission.
E16/E17 ablation | selesai 0.00s


,test_mean,test_pooled
svc,0.5006,0.5006
fusion_only,0.5223,0.5223
e16_selected,0.5242,0.5243


,comparison,ci95_low,ci95_high
0,fusion_only minus svc,0.0101,0.0333
1,e16_selected minus fusion_only,-0.0014,0.0053


## 5. E14 selesai dan E18 analisis kesalahan
E14 dibandingkan hanya pada fold0–2 terhadap kontrol yang identik, tanpa
membandingkan rerata3fold dengan5fold. CI bersyarat pada model historis dan tidak
mencakup bias seluruh proses research. Hasil E16 adalah cabang SVC+reader.

In [6]:
with step('audit E14 dan error slices', 1):
    audit=json.loads((ROOT/'research/experiments/e18_closing_audit/results.json').read_text())
    display(pd.DataFrame([dict(model=k,control=v['control_mean'],e14=v['e14_mean'],
                              ci_low=v['paired_prompt_ci95'][0],ci_high=v['paired_prompt_ci95'][1],
                              positive_folds=v['positive_folds']) for k,v in audit['e14'].items()]).round(4))
    print('E14 COMPLETE. OOF/log diunduh; source tidak menyimpan bobot tahap2. Tidak dipromosikan.')
    display(pd.DataFrame(audit['e16']['per_class']).round(4))
    display(pd.DataFrame(audit['e16']['length_slices']).round(4))
    display(pd.DataFrame(audit['e16']['top_confusions'][:6]))
    print('Fixes',audit['e16']['fixes'],'breaks',audit['e16']['breaks'])
    print('Family accuracy',audit['e16']['family_accuracy'],
          'fraction errors within family',audit['e16']['fraction_errors_within_family'])
    print('Slice adalah diagnostik pasca-test; bukan izin menyesuaikan kandidat pada test.')

audit E14 dan error slices | estimasi 1s
E14 COMPLETE. OOF/log diunduh; source tidak menyimpan bobot tahap2. Tidak dipromosikan.
Fixes 107 breaks 58
Family accuracy 0.7800869985500242 fraction errors within family 0.5376016260162602
Slice adalah diagnostik pasca-test; bukan izin menyesuaikan kandidat pada test.
audit E14 dan error slices | selesai 0.00s


,model,control,e14,ci_low,ci_high,positive_folds
0,q8,0.5686,0.5649,-0.0198,0.0130,1
1,ll8,0.5792,0.5705,-0.0265,0.0086,1


,model,n,svc_recall,e16_recall
0,claude-fable-5,114,0.2281,0.2281
1,claude-opus-4-6,115,0.7217,0.7826
2,claude-opus-4-8,115,0.3826,0.4696
3,claude-opus-5,115,0.6522,0.7130
4,claude-sonnet-5,115,0.6696,0.6609
5,gemini-3.1-pro-preview,115,0.8087,0.8174
6,gemini-3.6-flash,115,0.7217,0.6957
7,gemini-3.7-flash,115,0.1826,0.2261
8,gemini-3.8-flash,115,0.6783,0.6957
9,glm-5.3,115,0.2348,0.2783


,words,n,classes,svc_accuracy,e16_accuracy,e16_BA_present_classes
0,"(0.0, 64.0]",532,18,0.4041,0.4211,0.4098
1,"(64.0, 100.0]",504,18,0.4821,0.5159,0.5152
2,"(100.0, 160.0]",547,18,0.5137,0.5375,0.5341
3,"(160.0, inf]",486,18,0.6111,0.6317,0.6359


,true,predicted,n
0,gemini-3.7-flash,gemini-3.8-flash,68
1,gpt-5.6-sol,gpt-5.6-terra,26
2,kimi-k3,gpt-5.5,21
3,claude-fable-5,glm-5.3,21
4,glm-5.3-flash,glm-5.3,20
5,glm-5.3,glm-5.3-flash,20


## 6. Ekspektasi versus leaderboard resmi
Judul seperti “0.75+” bukan pengukuran. Jangan menghitung gap CV-LB jika CV yang
sebanding tidak tersedia. Skor private masih tidak diketahui. Data di bawah snapshot,
bukan polling live; cek ulang API sebelum tindakan baru.

In [7]:
with step('riwayat resmi', 1):
    history=json.loads((ROOT/'research/live/20261103/submissions.txt').read_text())
    rows=[]
    for r in history:
        rows.append(dict(submission=r['ref'],date=r['date'],status=r['status'].split('.')[-1],
                         public_BA=float(r['publicScore']) if r.get('publicScore') else None,
                         description=r['description']))
    history_table=pd.DataFrame(rows)
    display(history_table)
    print('Official best',history_table.public_BA.max())
    for ref,context in [(57889656,'target judul0.75+, tidak ada CV0.75 terverifikasi'),
                        (57919689,'penambahan komponen:LB0.661'),
                        (57919873,'perubahan multi-komponen:LB0.663')]:
        hit=history_table[history_table.submission==ref]
        if len(hit): print(ref,context,'actual',hit.public_BA.iloc[0])

riwayat resmi | estimasi 1s
Official best 0.665
57889656 target judul0.75+, tidak ada CV0.75 terverifikasi actual 0.639
57919689 penambahan komponen:LB0.661 actual 0.661
57919873 perubahan multi-komponen:LB0.663 actual 0.663
riwayat resmi | selesai 0.00s


,submission,date,status,public_BA,description
0,57930534,2026-11-03T03:11:20.083000,PENDING,NaN,NTL Apex-III Grandmaster SOTA (v3): 8-Model (q...
1,57919873,2026-11-02T19:46:02.997000,COMPLETE,0.663,NTL Apex-II Grandmaster SOTA (v2): Dual-DeBERT...
2,57919689,2026-11-02T19:37:07.900000,COMPLETE,0.661,Sovereign Grandmaster v2 (v3 Exception-Proof):...
3,57907202,2026-11-02T11:45:38.710000,COMPLETE,0.665,NTL Apex I SOTA Contender (190.5k Clean OpenPo...
4,57901383,2026-11-02T08:18:38.430000,COMPLETE,NaN,Sovereign Grandmaster v2: Restored Llama-3.1-8...
5,57889656,2026-11-02T00:47:10.843000,COMPLETE,0.639,SOTA Apex Grandmaster 0.75+ (198k Clean Pool +...
6,57876141,2026-11-01T15:25:29.167000,COMPLETE,0.664,fork of public One Swap 0.664 (zbigniewkirchne...
7,57874130,2026-11-01T14:10:06.747000,COMPLETE,0.659,SOTA Apex 0.72+ Full Multi-Model Ensemble With...
8,57873164,2026-11-01T13:34:23.267000,COMPLETE,0.330,S12 SOTA Apex Ensemble 0.72431 (v2 True Fixed ...
9,57872845,2026-11-01T13:22:41.957000,COMPLETE,NaN,S12 SOTA Apex Ensemble 0.72431 (Fixed Hidden R...


## 7. Komputasi, checkpoint, dan keputusan berikut

GPU akun yang sama habis; TPU masih tersedia pada snapshot. Sesi ini melakukan
eksperimen CPU singkat tanpa membuat runtime GPU/TPU interaktif baru. E14 telah
berjalan sebelum sesi ini; kini COMPLETE dan hasil/log sudah diunduh dan diaudit.
Jangan menggandakan job, menghentikan scoring resmi, atau beralih akun untuk melewati quota.

Berikutnya:replikasi fusion pada cache yang memiliki IDs/revision lengkap, uji pada
ensemble kuat dengan training nested yang benar, dan peroleh gold bertopik baru
untuk konfirmasi. Data exact-source bergaya host bernilai lebih daripada menambah
postprocessing kluster yang sudah gagal. Target0.99 masih belum didukung bukti.

In [8]:
with step('quota snapshot', 1):
    quota=json.loads((ROOT/'research/live/20261103/quota.txt').read_text())
    display(pd.DataFrame(quota))
    print('Handoff:readme.md; logs:research/experiments/e15_reader_foldlocal.log dan e16/e17.')
    print('Jalankan ulang training eksperimen hanya jika dibutuhkan; hasil selesai sudah disimpan.')

quota snapshot | estimasi 1s
Handoff:readme.md; logs:research/experiments/e15_reader_foldlocal.log dan e16/e17.
Jalankan ulang training eksperimen hanya jika dibutuhkan; hasil selesai sudah disimpan.
quota snapshot | selesai 0.00s


,resource,used,remaining,total,refreshAt
0,GPU,31.85h,0.00h,30.00h,2026-11-07T00:00:00+00:00
1,TPU,0.00h,20.00h,20.00h,2026-11-07T00:00:00+00:00
